# Lenormand B7 — Top-8 Sprint（CPU，无重训）

线上锚点：**Subtask 1 = 0.7854，Subtask 2 = 0.6364，Composite = 0.7407**。当前第八名约为 **0.7629**。

这个 notebook 不加载、训练或推理任何 14B/27B 模型，只使用 B4 已保存的三折 OOF/test logits 与 B4-E2 candidate-meta 表，预计普通 Colab CPU **1–4 分钟**：

1. 在严格 grouped outer OOF 上检验 ACC label-shift Factor decoder；没有过 gate 就禁止部署。
2. 生成一个只改 Factor 的 fold-margin 分布漂移探针。
3. 生成两个只改 Evidence 的阈值探针：Recall-0.35 与 Precision-0.45。
4. 自动验证列名、行序、Factor taxonomy、Indicator 空证据、证据逐字出处、组件隔离与 SHA256。
5. 输出三个独立目录，每个目录里都有可直接上传的 `Lenormand.csv`。

注意：Evidence 0.35/0.45 是**公开榜诊断探针**，不是重新声称通过了隐藏验证。三次提交严格一次只改变一个组件，后续才能把有效增益组合起来。


In [ ]:
#@title 1. Drive、路径与运行开关
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast, importlib, json, shutil, sys
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
TEST_PATH = ROOT / 'leaderboard.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH = ROOT / 'ieee/train.xlsx'
if not TEST_PATH.exists(): TEST_PATH = ROOT / 'ieee/leaderboard.xlsx'

Q14_OOF = ROOT / 'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
Q38_OOF = ROOT / 'results/B4_Q38F_FULL64_THREE_FOLD_OOF/Q38_FULL64_OOF.npz'
FINAL_ROOT = ROOT / 'results/B4_FINAL_SUBMISSION'
FACTOR_TEST_ROOT = FINAL_ROOT / 'FACTOR_TEST'
ORIGINAL_SUBMISSION = FINAL_ROOT / 'SUBMISSION/Lenormand.csv'
META_TEST = FINAL_ROOT / 'TASK1_TEST/FINAL/task1_test_candidate_meta_audit.csv'

Q14_TEST = [FACTOR_TEST_ROOT / f'Q14/fold_{fold}/factor_test_logits.npz' for fold in range(3)]
Q38_TEST = [FACTOR_TEST_ROOT / f'Q38/fold_{fold}/factor_test_logits.npz' for fold in range(3)]
OUT = ROOT / 'results/B7_TOP8_SPRINT'
OUT.mkdir(parents=True, exist_ok=True)

RUN_FACTOR_OOF = True
BUILD_THREE_PROBES = True

required = [TRAIN_PATH, TEST_PATH, Q14_OOF, Q38_OOF, ORIGINAL_SUBMISSION, META_TEST, *Q14_TEST, *Q38_TEST]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少 B4 冻结缓存；B7 不会重跑大模型：\n' + '\n'.join(missing))
if not (ROOT / 'b7_top8_sprint.py').exists():
    raise FileNotFoundError('请先把 b7_top8_sprint.py 放到 ' + str(ROOT))
sys.path.insert(0, str(ROOT))
print({'output': str(OUT), 'gpu_needed': False, 'required_artifacts': len(required)})


In [ ]:
#@title 2. 导入 B7 并冻结配置
import b7_top8_sprint as b7
importlib.reload(b7)
assert b7.B7_RUNTIME_REVISION == '2026-08-27.top8-sprint-v1'

FACTOR_CFG = b7.FactorQuantificationConfig(
    q38_weight=0.75,
    acc_shrinks=(0.25, 0.50),
    gate_macro_f1=0.006,
    gate_tail_macro_f1=0.0,
    gate_folds_improved=2,
    gate_worst_fold_delta=-0.005,
)
EVIDENCE_DEFAULT = b7.EvidenceProbeConfig(probability_threshold=0.40)
EVIDENCE_RECALL = b7.EvidenceProbeConfig(probability_threshold=0.35)
EVIDENCE_PRECISION = b7.EvidenceProbeConfig(probability_threshold=0.45)

print({
    'revision': b7.B7_RUNTIME_REVISION,
    'factor_gate': {
        'macro_f1_delta': FACTOR_CFG.gate_macro_f1,
        'tail_delta': FACTOR_CFG.gate_tail_macro_f1,
        'folds_improved': FACTOR_CFG.gate_folds_improved,
    },
    'evidence_thresholds': [0.35, 0.40, 0.45],
})


In [ ]:
#@title 3. 硬对齐训练/测试、Q14/Q38 三折概率
train = pd.read_excel(TRAIN_PATH, dtype=str).fillna('')
test = pd.read_excel(TEST_PATH, dtype=str).fillna('')
original = pd.read_csv(ORIGINAL_SUBMISSION, dtype=str, keep_default_na=False)
assert original.columns.tolist() == list(b7.OFFICIAL_COLUMNS)
assert original.row_id.astype(str).tolist() == test.row_id.astype(str).tolist()

q14_saved = np.load(Q14_OOF, allow_pickle=True)
q38_saved = np.load(Q38_OOF, allow_pickle=True)
row_ids = q14_saved['row_ids'].astype(str)
folds = q14_saved['folds'].astype(int)
targets = q14_saved['targets'].astype(np.int8)
assert q38_saved['row_ids'].astype(str).tolist() == row_ids.tolist()
assert np.array_equal(q38_saved['folds'].astype(int), folds)
assert np.array_equal(q38_saved['targets'].astype(np.int8), targets)
assert train.row_id.astype(str).tolist() == row_ids.tolist()

q14_key = 'verifier_logits' if 'verifier_logits' in q14_saved.files else 'logits'
q38_key = 'logits' if 'logits' in q38_saved.files else 'verifier_logits'
q14_probability = b7.sigmoid(q14_saved[q14_key])
q38_probability = b7.sigmoid(q38_saved[q38_key])
oof_probability = (
    (1.0 - FACTOR_CFG.q38_weight) * q14_probability
    + FACTOR_CFG.q38_weight * q38_probability
).astype(np.float32)

test_row_ids = test.row_id.astype(str).to_numpy()
test_fold_probabilities = []
for fold in range(3):
    q14 = b7.load_probability_npz(Q14_TEST[fold], test_row_ids, ('logits','verifier_logits','probabilities'))
    q38 = b7.load_probability_npz(Q38_TEST[fold], test_row_ids, ('logits','verifier_logits','probabilities'))
    test_fold_probabilities.append(
        ((1.0 - FACTOR_CFG.q38_weight) * q14 + FACTOR_CFG.q38_weight * q38).astype(np.float32)
    )

print({
    'train_rows': len(row_ids), 'test_rows': len(test_row_ids),
    'fold_sizes': np.bincount(folds).tolist(),
    'factor_support_min_max': [int(targets.sum(0).min()), int(targets.sum(0).max())],
})


In [ ]:
#@title 4. 严格 grouped outer-OOF：ACC 量化 gate
factor_decision, factor_state = b7.run_factor_quantification_oof(
    row_ids=row_ids,
    folds=folds,
    target=targets,
    probability=oof_probability,
    config=FACTOR_CFG,
    output_dir=OUT / 'FACTOR_OOF',
)

factor_summary = pd.read_csv(OUT / 'FACTOR_OOF/B7_FACTOR_OOF_SUMMARY.csv')
factor_gate = pd.read_csv(OUT / 'FACTOR_OOF/B7_FACTOR_GATE.csv')
factor_folds = pd.read_csv(OUT / 'FACTOR_OOF/B7_FACTOR_OUTER_FOLDS.csv')
display(factor_summary.sort_values('macro_f1', ascending=False))
display(factor_gate)
display(factor_folds.pivot(index='outer_fold', columns='system', values='macro_f1'))

if factor_decision['factor_gate_passed']:
    print('ACC GATE PASS：允许它成为 Slot-1 Factor 候选。')
else:
    print('ACC GATE FAIL：禁止把 ACC 包装成升级；Slot-1 使用可归因的 fold-margin 分布漂移探针。')


In [ ]:
#@title 5. 复现已上传 Factor，并生成唯一 Slot-1 Factor 探针
factor_systems, test_prior_audit = b7.deploy_factor_decoders(
    oof_probability=oof_probability,
    target=targets,
    test_fold_probabilities=test_fold_probabilities,
    state=factor_state,
    factor_decision=factor_decision,
    config=FACTOR_CFG,
)
if len(test_prior_audit):
    test_prior_audit.to_csv(OUT / 'FACTOR_TEST_PRIOR_AUDIT.csv', index=False)

original_factor_sets = original.factors.map(lambda value: set(ast.literal_eval(value))).tolist()
recreated_sets = [
    {b7.FACTOR_LABELS[label] for label in np.flatnonzero(row)}
    for row in factor_systems['CURRENT_RECREATED']
]
factor_reproduction = float(np.mean([left == right for left, right in zip(original_factor_sets, recreated_sets)]))
if factor_reproduction != 1.0:
    raise AssertionError(f'B7 无法逐行复现已上传 Factor，停止：agreement={factor_reproduction:.6f}')

if factor_decision['factor_gate_passed']:
    slot1_system = factor_decision['selected_factor_system']
    slot1_name = '01_FACTOR_' + slot1_system
    slot1_policy = 'STRICT_OOF_LICENSED'
else:
    slot1_system = 'FOLD_MARGIN_PROBE'
    slot1_name = '01_FACTOR_FOLD_MARGIN_SHIFT_PROBE'
    slot1_policy = 'PUBLIC_SHIFT_DIAGNOSTIC_ONLY'

slot1_factor = b7.factor_frame(test_row_ids, factor_systems[slot1_system])
current = factor_systems['CURRENT_RECREATED']
slot1_pred = factor_systems[slot1_system]
print({
    'factor_reproduction': factor_reproduction,
    'slot1_system': slot1_system,
    'slot1_policy': slot1_policy,
    'mean_factors_current': float(current.sum(1).mean()),
    'mean_factors_slot1': float(slot1_pred.sum(1).mean()),
    'changed_rows': int(np.any(current != slot1_pred, axis=1).sum()),
    'changed_label_cells': int((current != slot1_pred).sum()),
})


In [ ]:
#@title 6. 精确复现 B4-E2 Evidence，再做 0.35 / 0.45 单变量探针
candidate_meta = pd.read_csv(META_TEST, keep_default_na=False)
risk_frame = original[['row_id','risk_level']].copy()

default_by_row = b7.decode_evidence_candidates(candidate_meta, risk_frame, EVIDENCE_DEFAULT)
default_task1 = b7.evidence_frame(risk_frame, default_by_row)
evidence_equal = default_task1.evidence.astype(str).to_numpy() == original.evidence.astype(str).to_numpy()
if not evidence_equal.all():
    mismatch = pd.DataFrame({
        'row_id': original.loc[~evidence_equal, 'row_id'],
        'uploaded': original.loc[~evidence_equal, 'evidence'],
        'recreated': default_task1.loc[~evidence_equal, 'evidence'],
    })
    mismatch.to_csv(OUT / 'EVIDENCE_REPRODUCTION_FAILURE.csv', index=False)
    raise AssertionError(
        f'默认 0.40 decoder 未逐行复现已上传 Evidence（{len(mismatch)} rows）；停止，不能做不可归因探针。'
    )

recall_by_row = b7.decode_evidence_candidates(candidate_meta, risk_frame, EVIDENCE_RECALL)
precision_by_row = b7.decode_evidence_candidates(candidate_meta, risk_frame, EVIDENCE_PRECISION)
recall_task1 = b7.evidence_frame(risk_frame, recall_by_row)
precision_task1 = b7.evidence_frame(risk_frame, precision_by_row)

evidence_probe_summary = pd.DataFrame([
    {
        'system': 'UPLOADED_DEFAULT_040',
        'threshold': 0.40,
        'mean_phrases': original.evidence.map(lambda x: len([p for p in str(x).split(';') if p.strip()])).mean(),
        'changed_rows_vs_uploaded': 0,
    },
    {
        'system': 'EVIDENCE_RECALL_035',
        'threshold': 0.35,
        'mean_phrases': recall_task1.evidence.map(lambda x: len([p for p in str(x).split(';') if p.strip()])).mean(),
        'changed_rows_vs_uploaded': int((recall_task1.evidence != original.evidence).sum()),
    },
    {
        'system': 'EVIDENCE_PRECISION_045',
        'threshold': 0.45,
        'mean_phrases': precision_task1.evidence.map(lambda x: len([p for p in str(x).split(';') if p.strip()])).mean(),
        'changed_rows_vs_uploaded': int((precision_task1.evidence != original.evidence).sum()),
    },
])
display(evidence_probe_summary)
evidence_probe_summary.to_csv(OUT / 'B7_EVIDENCE_PROBE_SUMMARY.csv', index=False)


In [ ]:
#@title 7. 写出 3 个组件隔离的官方提交 + 硬审计
audits = []
if BUILD_THREE_PROBES:
    audits.append(b7.write_submission_variant(
        name=slot1_name,
        original=original,
        test_frame=test,
        output_dir=OUT / 'PROBES',
        factor_prediction_frame=slot1_factor,
        declared_component='FACTOR_ONLY',
    ))
    audits.append(b7.write_submission_variant(
        name='02_EVIDENCE_RECALL_035',
        original=original,
        test_frame=test,
        output_dir=OUT / 'PROBES',
        task1_frame=recall_task1,
        declared_component='EVIDENCE_ONLY',
    ))
    audits.append(b7.write_submission_variant(
        name='03_EVIDENCE_PRECISION_045',
        original=original,
        test_frame=test,
        output_dir=OUT / 'PROBES',
        task1_frame=precision_task1,
        declared_component='EVIDENCE_ONLY',
    ))

audit_frame = pd.DataFrame(audits)
display(audit_frame[[
    'candidate','declared_component','rows','mean_evidence_phrases','mean_factors',
    'changed_risk_level_rows','changed_evidence_rows','changed_factors_rows',
    'changed_any_rows','sha256','path','status',
]])

# Component isolation is a hard stop, not a warning.
factor_audit = audits[0]
assert factor_audit['changed_risk_level_rows'] == 0 and factor_audit['changed_evidence_rows'] == 0
for evidence_audit in audits[1:]:
    assert evidence_audit['changed_risk_level_rows'] == 0 and evidence_audit['changed_factors_rows'] == 0
assert all(row['indicator_nonempty_evidence'] == 0 for row in audits)
assert all(row['verbatim_failures'] == 0 for row in audits)

upload_order = pd.DataFrame([
    {
        'slot': 1, 'candidate': slot1_name, 'changes': 'Factor only',
        'observe': 'Subtask 2 delta only',
        'interpretation': slot1_policy,
        'path': audits[0]['path'], 'sha256': audits[0]['sha256'],
    },
    {
        'slot': 2, 'candidate': '02_EVIDENCE_RECALL_035', 'changes': 'Evidence only',
        'observe': 'Subtask 1 delta only',
        'interpretation': 'If positive, current decoder is too conservative',
        'path': audits[1]['path'], 'sha256': audits[1]['sha256'],
    },
    {
        'slot': 3, 'candidate': '03_EVIDENCE_PRECISION_045', 'changes': 'Evidence only',
        'observe': 'Subtask 1 delta only',
        'interpretation': 'If positive, current decoder is too permissive',
        'path': audits[2]['path'], 'sha256': audits[2]['sha256'],
    },
])
upload_order.to_csv(OUT / 'B7_UPLOAD_ORDER.csv', index=False)
display(upload_order)


In [ ]:
#@title 8. 决策文件、打包与下载路径
decision = {
    'version': 'B7-TOP8-SPRINT-v1',
    'runtime_revision': b7.B7_RUNTIME_REVISION,
    'online_anchor': {'subtask1': 0.7854, 'subtask2': 0.6364, 'composite': 0.7407},
    'target_reference': {'rank8_composite_seen_2026_08_27': 0.7629},
    'factor_oof_decision': factor_decision,
    'slot1_system': slot1_system,
    'slot1_policy': slot1_policy,
    'evidence_default_reproduced_exactly': True,
    'candidate_audits': audits,
    'upload_rule': (
        'Upload all three only as isolated public-leaderboard probes. Do not combine them until '
        'the component-specific online deltas are observed. A separate final hidden set means '
        'a one-off public gain should still be treated conservatively.'
    ),
}
b7.json_dump(decision, OUT / 'B7_DECISION.json')

package = OUT / 'UPLOAD_PACKAGE'
if package.exists(): shutil.rmtree(package)
package.mkdir(parents=True)
for source in [
    OUT / 'B7_DECISION.json', OUT / 'B7_UPLOAD_ORDER.csv',
    OUT / 'B7_EVIDENCE_PROBE_SUMMARY.csv',
    OUT / 'FACTOR_OOF/B7_FACTOR_OOF_SUMMARY.csv',
    OUT / 'FACTOR_OOF/B7_FACTOR_GATE.csv',
    OUT / 'FACTOR_OOF/B7_FACTOR_OUTER_FOLDS.csv',
]:
    shutil.copy2(source, package / source.name)
shutil.copytree(OUT / 'PROBES', package / 'PROBES')
archive = shutil.make_archive('/content/Lenormand_B7_TOP8_SPRINT', 'zip', package)

print('\n=== UPLOAD IN THIS ORDER ===')
for row in upload_order.itertuples(index=False):
    print(f'{row.slot}. {row.candidate}')
    print('   ', row.path)
    print('   SHA256:', row.sha256)
print('\nPackage:', archive)
print('上传的是各候选目录内的 Lenormand.csv，不是 zip。zip 只用于把结果发回给我审计。')
# from google.colab import files
# files.download(archive)


In [ ]:
#@title 9. 线上三次结果回来后，填数自动决定下一步（可选）
# 只填对应探针改变的分项；没有结果保持 None。
ONLINE_FACTOR_PROBE_SUBTASK2 = None
ONLINE_EVIDENCE_RECALL_SUBTASK1 = None
ONLINE_EVIDENCE_PRECISION_SUBTASK1 = None

observations = []
if ONLINE_FACTOR_PROBE_SUBTASK2 is not None:
    observations.append({
        'component': 'Factor', 'candidate': slot1_name,
        'online_score': float(ONLINE_FACTOR_PROBE_SUBTASK2),
        'anchor': 0.6364,
        'delta': float(ONLINE_FACTOR_PROBE_SUBTASK2) - 0.6364,
    })
for candidate, score in [
    ('EVIDENCE_RECALL_035', ONLINE_EVIDENCE_RECALL_SUBTASK1),
    ('EVIDENCE_PRECISION_045', ONLINE_EVIDENCE_PRECISION_SUBTASK1),
]:
    if score is not None:
        observations.append({
            'component': 'Task1/Evidence', 'candidate': candidate,
            'online_score': float(score), 'anchor': 0.7854,
            'delta': float(score) - 0.7854,
        })
if observations:
    observation_frame = pd.DataFrame(observations).sort_values('delta', ascending=False)
    display(observation_frame)
    print('把这张表发回；下一版只组合 delta>0 且方向可解释的组件。')
else:
    print('等待三个隔离探针的线上分项结果。')
